# MAC와 자료형

계산 횟수를 세고, FP32·FP16·INT8의 저장량과 표현 차이를 확인합니다.

In [1]:
import math
import numpy as np
import torch
import matplotlib.pyplot as plt
import embedded_lab as lab
from embedded_lab import table

lab.start()

## 1. MAC
MAC는 곱한 값을 누적하는 계산입니다. 성능을 표시할 때 **1 MAC = 곱셈 1회 + 덧셈 1회 = 2 operations**로 셉니다.
`2×1 + 3×2 + 1×3 + 4×3`은 4 MAC입니다. 덧셈 기호는 3개지만, 여기서는 누적값 0부터 더하는 기준을 사용합니다.

In [2]:
values = [2, 3, 1, 4]
weights = [1, 2, 3, 3]
total = 0
rows = []
for value, weight in zip(values, weights):
    before = total
    total += value * weight
    rows.append([f'{before} + {value} × {weight}', total])
table(['계산', '누적값'], rows)

계산,누적값
0 + 2 × 1,2
2 + 3 × 2,8
8 + 1 × 3,11
11 + 4 × 3,23


### ✏️ A01. MAC → operations → GOP
**1 GOP = 10⁹ operations**입니다. MAC 수를 받아 `(operations, GOP)`를 반환하세요.

In [5]:
def mac_to_ops(mac):
    ops = 2* mac
    return ops, ops/1e9
    #raise NotImplementedError('A01 함수를 완성하세요.')

In [6]:
assert mac_to_ops(4) == (8, 8e-9)
assert mac_to_ops(0) == (0, 0)
assert mac_to_ops(500_000_000) == (1_000_000_000, 1.0)
print('A01 완료')

A01 완료


## 2. 저장량
FP32는 값 하나에 4바이트, FP16은 2바이트, INT8은 1바이트를 사용합니다.
같은 개수의 값을 저장해 비교합니다. 아래 MB는 10⁶바이트 기준입니다.

In [7]:
rows = []
for dtype in (np.float32, np.float16, np.int8):
    values = np.zeros(1_000_000, dtype=dtype)
    rows.append([np.dtype(dtype).name, values.itemsize, values.nbytes / 1e6])
table(['자료형', '값 하나의 바이트', '100만 개의 용량 MB'], rows)

자료형,값 하나의 바이트,100만 개의 용량 MB
float32,4,4.0
float16,2,2.0
int8,1,1.0


**확인:** FP32·FP16·INT8의 저장량 비율은 얼마인가요?

**답:**

## 3. FP32와 FP16의 표현 오차
부동소수점은 표현 가능한 가까운 값으로 수를 저장합니다. 아래에서는 더 정밀한 float64 값을 기준으로 오차를 비교합니다.

In [8]:
reference = np.array([0.1, 1.23, -0.53, 12.345], dtype=np.float64)
f32 = reference.astype(np.float32).astype(np.float64)
f16 = reference.astype(np.float16).astype(np.float64)
table(['기준값', 'FP32 값', 'FP32 오차', 'FP16 값', 'FP16 오차'],
      [[v, f'{a:.9g}', f'{abs(a-v):.3g}', f'{b:.9g}', f'{abs(b-v):.3g}']
       for v, a, b in zip(reference, f32, f16)])

기준값,FP32 값,FP32 오차,FP16 값,FP16 오차
0.1,0.100000001,1.49e-09,0.0999755859,2.44e-05
1.23,1.23000002,1.91e-08,1.23046875,0.000469
-0.53,-0.529999971,2.86e-08,-0.529785156,0.000215
12.345,12.3450003,2.67e-07,12.34375,0.00125


## 4. INT8과 스케일
INT8은 -128~127의 정수를 저장합니다. **스케일 0.1**을 사용하면 정수 12가 실제 값 1.2를 뜻합니다.
1.23은 `1.23÷0.1 → 반올림 → 12`로 저장하고, `12×0.1 → 1.2`로 복원합니다. 이 예제의 복원 범위는 -12.8~12.7입니다.

In [9]:
values = np.array([0.1, 1.23, -0.53, 12.345], dtype=np.float32)
scale = 0.1
stored = np.clip(np.rint(values / scale), -128, 127).astype(np.int8)
restored = stored.astype(np.float32) * scale
table(['원래 값', '저장 정수', '복원값', '오차'],
      [[round(float(v), 6), int(q), round(float(r), 6), round(float(abs(v-r)), 6)]
       for v, q, r in zip(values, stored, restored)])

원래 값,저장 정수,복원값,오차
0.1,1,0.1,0.0
1.23,12,1.2,0.03
-0.53,-5,-0.5,0.03
12.345,123,12.3,0.045


**확인:** -0.53은 어떤 정수로 저장되고 어떤 값으로 복원되나요?

**답:**  저장정수 05, 복원값 -0.5 

여기서는 배열의 저장량과 표현 오차를 비교했습니다. CNN의 정확도와 추론 속도는 모델을 실행해 별도로 측정합니다.